In [22]:
import os
import json
from collections import Counter
from pathlib import Path
import numpy as np
import torch
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from vul_detector import VulDetector
from vul_trainer import VulTrainerManual
from imblearn.over_sampling import SMOTE
import time
from torch.utils.data import WeightedRandomSampler

In [11]:
import time
start = time.time()
time.sleep(5)
end = time.time()
print(f"Elapsed time: {end - start} seconds")

Elapsed time: 5.000445365905762 seconds


In [23]:
# -------------------------
# 0) Load JSONL -> HF Dataset
# -------------------------
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

# Set this to your folder containing train/val/test jsonl from preprocessing
DATASET_DIR = os.path.join(os.getcwd(), "data")
assert os.path.isdir(DATASET_DIR), f"Dataset dir not found: {DATASET_DIR}"

train_dataset = Dataset.from_list(load_jsonl(os.path.join(DATASET_DIR, "train.jsonl")))
valid_dataset = Dataset.from_list(load_jsonl(os.path.join(DATASET_DIR, "val.jsonl")))
test_dataset = Dataset.from_list(load_jsonl(os.path.join(DATASET_DIR, "test.jsonl")))

print("Loaded:", len(test_dataset))
print("Example row keys:", test_dataset.column_names)
print("Example:", {k: test_dataset[0][k] for k in ["id", "project", "target", "answer_text"]})

Loaded: 26300
Example row keys: ['id', 'project', 'target', 'func_clean', 'prompt', 'answer_text']
Example: {'id': '6e74b9161699646e', 'project': 'ghostscript', 'target': 1, 'answer_text': 'vulnerable'}


In [3]:
sum(1 for x in test_dataset['target'] if x == 1)

846

In [4]:

sum(1 for x in test_dataset['target'] if x == 0)

21971

In [24]:
# -------------------------
# 2) Tokenizer + Map
# -------------------------
model_name = "microsoft/unixcoder-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

MAX_LEN = 1024  # start 256; try 384 if needed (1024 will likely OOM)

# def tokenize_batch(batch):
#     texts = [c.strip() for c in batch["func_clean"]]

#     enc = tokenizer(
#         texts,
#         truncation=True,
#         max_length=MAX_LEN,
#         padding="max_length",
#         add_special_tokens=True,
#     )

#     # Labels: prefer numeric target
#     if "target" in batch:
#         enc["labels"] = [int(x) for x in batch["target"]]
#     else:
#         enc["labels"] = [
#             1 if a.strip().lower() == "vulnerable" else 0
#             for a in batch["answer_text"]
#         ]
#     return enc
def tokenize_batch(batch):
    texts = [c.strip() for c in batch["func_clean"]]
    enc = tokenizer(texts, truncation=True, max_length=MAX_LEN, padding="max_length")

    # debug: detect truncation
    # if tokenizer returns overflowing info, use return_overflowing_tokens=True
    # easiest: compare pre-length vs MAX_LEN
    raw_lens = [len(tokenizer(t).input_ids) for t in texts]
    trunc_rate = sum(l > MAX_LEN for l in raw_lens) / len(raw_lens)
    print("trunc_rate:", trunc_rate)

    enc["labels"] = [int(x) for x in batch["target"]]
    return enc
train_tok = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names,
)
valid_tok = valid_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=valid_dataset.column_names,
)
test_tok = test_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=test_dataset.column_names,
)

Map:   0%|          | 0/110878 [00:00<?, ? examples/s]

trunc_rate: 0.187


Map:   1%|          | 1000/110878 [00:01<03:05, 591.98 examples/s]

trunc_rate: 0.144


Map:   2%|▏         | 2000/110878 [00:03<02:42, 668.60 examples/s]

...

trunc_rate: 0.028


Map:  95%|█████████▌| 25000/26300 [00:32<00:01, 865.40 examples/s]

trunc_rate: 0.094


Map: 100%|██████████| 26300/26300 [00:34<00:00, 769.78 examples/s]

trunc_rate: 0.01


In [6]:

# compute embeddings for train set, apply SMOTE to embeddings, produce a balanced DataLoader

def balance_embeddings_with_smote(batcher, emb_model, embed_device):
    emb_model.to(embed_device)
    emb_model.eval()
    embs = []
    ys = []
    with torch.no_grad():
        for b in batcher:
            input_ids = b["input_ids"].to(embed_device)
            attention_mask = b["attention_mask"].to(embed_device)
            out = emb_model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
            # use pooler_output if available else mean-pool last hidden state
            if hasattr(out, "pooler_output") and out.pooler_output is not None:
                pooled = out.pooler_output
            else:
                pooled = out.last_hidden_state.mean(dim=1)
            embs.append(pooled.cpu())
            ys.append(b["labels"].cpu())

    X = torch.cat(embs).numpy()
    y = torch.cat(ys).numpy()

    print("Before SMOTE:", Counter(y))

    smote = SMOTE(sampling_strategy="auto", k_neighbors=3, random_state=42)
    X_smote, y_smote = smote.fit_resample(X, y)

    print("After SMOTE:", Counter(y_smote))

    # create a DataLoader of embeddings+labels (for training a classifier on embeddings)
    X_smote_t = torch.tensor(X_smote, dtype=torch.float32)
    y_smote_t = torch.tensor(y_smote, dtype=torch.long)
    smote_dataset = torch.utils.data.TensorDataset(X_smote_t, y_smote_t)
    smote_loader = DataLoader(smote_dataset, batch_size=32, shuffle=True)
    return smote_loader


In [26]:
train_dataset[0]["target"]

1

In [43]:
# -------------------------
# 3) DataLoaders + class weights
# -------------------------
# embed_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Embedding device:", embed_device)
# emb_model = AutoModel.from_pretrained(model_name)

columns = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=columns)
valid_tok.set_format(type="torch", columns=columns)
test_tok.set_format(type="torch", columns=columns)

labels = np.array(train_dataset["target"]).astype(np.int64)
label_counts = Counter(test_dataset["target"])
total = sum(label_counts.values())
class_weights = [total / (2 * label_counts[i]) for i in range(2)]
sample_weights = np.array([class_weights[int(l)] for l in labels])

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_tok, batch_size=16, sampler=sampler)
val_loader = DataLoader(valid_tok, batch_size=32)
test_loader = DataLoader(test_tok, batch_size=32)

# balance_train_loader = balance_embeddings_with_smote(train_loader, emb_model, embed_device)
# balance_val_loader = balance_embeddings_with_smote(val_loader, emb_model, embed_device)
# balance_test_loader = balance_embeddings_with_smote(test_loader, emb_model, embed_device)

print("Class weights:", class_weights)
print("Batches -> train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))

Class weights: [0.5142946536821933, 17.9890560875513]
Batches -> train: 6930 val: 2723 test: 822


In [44]:
for batch in train_loader:
    inputs = batch["input_ids"]
    labels = batch["labels"]
    print("Sample batch shapes:", inputs.shape, labels.shape)
    
    # Count positive and negative labels
    num_positive = (labels == 1).sum().item()
    num_negative = (labels == 0).sum().item()
    print(f"Positive (vulnerable): {num_positive}")
    print(f"Negative (non-vulnerable): {num_negative}")
    break

Sample batch shapes: torch.Size([16, 1024]) torch.Size([16])
Positive (vulnerable): 9
Negative (non-vulnerable): 7


In [ ]:
# -------------------------
# 4) Train vulnerability detector
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = VulDetector(model_name=model_name, num_labels=2)
trainer = VulTrainerManual(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    class_weights=class_weights,
    learning_rate=1e-5,
    num_epochs=12,
    loss_type="focal",
    focal_gamma=1.5,
)

#trainer.train()

Using device: cuda
1026


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 559.83it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: microsoft/unixcoder-base
Key                        | Status     | 
---------------------------+------------+-
embeddings.position_ids    | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NameError: name 'val_loader' is not defined

In [ ]:
# -------------------------
# 5) Evaluate on validation/test splits
# -------------------------
def evaluate_loader(loader):
    model.eval()
    total_loss = 0.0
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels_batch = batch.pop("labels")
            outputs = model(**batch)
            logits = outputs.logits
            loss = trainer.criterion(logits, labels_batch)
            total_loss += loss.item()
            preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            labels.extend(labels_batch.cpu().tolist())
    metrics = trainer.compute_metrics(np.array(preds), np.array(labels))
    metrics["loss"] = total_loss / max(len(loader), 1)
    return metrics

best_ckpts = sorted(Path(".").glob("best_model_epoch_*.pt"), key=lambda p: p.stat().st_mtime)
if best_ckpts:
    best_ckpt = best_ckpts[-1]
    model.load_state_dict(torch.load(best_ckpt, map_location=device))
    model.to(device)
    print(f"Loaded best checkpoint: {best_ckpt}")
else:
    print("No saved checkpoints found; evaluating current model state.")

val_metrics = evaluate_loader(val_loader)
print("Validation metrics:", val_metrics)

test_metrics = evaluate_loader(test_loader)
print("Test metrics:", test_metrics)